This notebook downloads preliminary Sentinel-1 and Sentinel-2 global mosaic data for each tile in the 50x50 km grid. Data is downloaded using Copernicus Data Space Ecosystem (CDSE) STAC API. 

In [ ]:
import os
import sys
from urllib3 import Retry

import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
import rasterio
from rasterio.enums import Resampling

import stackstac
import pystac_client
from pystac_client.stac_api_io import StacApiIO

from dask.distributed import Client, LocalCluster


parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

KEY_ID = os.getenv("AWS_ACCESS_KEY_ID", "")
SA_KEY = os.getenv("AWS_SECRET_ACCESS_KEY", "")

try:
    from config_local import KEY_ID as KEY_ID_LOCAL, SA_KEY as SA_KEY_LOCAL
    if KEY_ID_LOCAL:
        KEY_ID = KEY_ID_LOCAL
    if SA_KEY_LOCAL:
        SA_KEY = SA_KEY_LOCAL
    print("Loaded local config")
except ImportError:
    print("No local config found; using environment variables or defaults")

In [ ]:
os.environ["GDAL_HTTP_TCP_KEEPALIVE"] = "YES"
os.environ["AWS_S3_ENDPOINT"] = "eodata.dataspace.copernicus.eu"
os.environ["AWS_ACCESS_KEY_ID"] = KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = SA_KEY
os.environ["AWS_HTTPS"] = "YES"
os.environ["AWS_VIRTUAL_HOSTING"] = "FALSE"
os.environ["GDAL_HTTP_UNSAFESSL"] = "YES"

gdal_env = stackstac.DEFAULT_GDAL_ENV.updated({
    "GDAL_NUM_THREADS": -1,
    "GDAL_HTTP_UNSAFESSL": "YES",
    "GDAL_HTTP_TCP_KEEPALIVE": "YES",
    "AWS_VIRTUAL_HOSTING": "FALSE",
    "AWS_HTTPS": "YES",
})

In [ ]:
# set CDSE STAC API link and retry strategy for STAC API requests
stac_url = 'https://stac.dataspace.copernicus.eu/v1/'

retry = Retry(
    total=5, backoff_factor=1, status_forcelist=[502, 503, 504], allowed_methods=None
)
stac_api_io = StacApiIO(max_retries=retry)
stac = pystac_client.Client.open(stac_url, stac_io=stac_api_io)

In [ ]:
# create Dask cluster and client
cluster = LocalCluster(processes=True, n_workers=16, memory_limit='30GB')
client = Client(cluster)
print(cluster.dashboard_link)

In [ ]:
# set variables
tile_path = "../data/grid_50km_epsg3059.geojson"

years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

In [ ]:
# load tiles and reproject to WGS84
tiles = gpd.read_file(tile_path)
tiles = tiles.to_crs(4326)

In [ ]:
# iteratively process each year and tile for Sentinel-1 and Sentinel-2 data
for year in years:
    outfolder_s1 = f"../data/grasslvnd/s1_{year}"
    outfolder_s2 = f"../data/grasslvnd/s2_{year}"

    # create output directories, set environment variables
    os.makedirs(outfolder_s1, exist_ok=True)
    os.makedirs(outfolder_s2, exist_ok=True)

    # Sentinel-1 data extraction
    bands = ['VV', 'VH']

    for idx, tile in tiles.iterrows():
        try:
            search = stac.search(
                collections=['sentinel-1-global-mosaics'],
                datetime=f"{year}-04-01T00:00:00Z/{year}-09-30T23:59:59Z",
                intersects=tile.geometry.__geo_interface__,
                limit=1000
            )

            items = search.item_collection()
            if len(items) == 0:
                print(f"no Sentinel-1 data found for tile {idx}, skip.")
                continue

            bounds_3059 = gpd.GeoSeries([tile.geometry], crs=4326).to_crs(3059).total_bounds.tolist()

            # stack both VV and VH at once
            stack = stackstac.stack(
                items=items,
                assets=bands,
                resolution=10,
                bounds=bounds_3059,
                snap_bounds=True,
                resampling=Resampling.bilinear,
                xy_coords='center',
                chunksize=1024,
                epsg=3059,
                gdal_env=gdal_env
            )

            # resample by month start
            stack = stack.resample(time="MS").first("time", keep_attrs=True)

            # process each band separately
            for band in bands:
                output_path = os.path.join(outfolder_s1, f"cdse_s1_{band.lower()}_tile_{idx}.tif")
                if os.path.exists(output_path):
                    print(f"{idx} band {band} already exists, skip.")
                    continue
                
                band_stack = stack.sel(band=band)
                band_stack = band_stack.transpose('time', 'y', 'x').compute()
                band_stack.rio.write_crs("EPSG:3059", inplace=True)

                output_path = os.path.join(outfolder_s1, f"cdse_s1_{band.lower()}_tile_{idx}.tif")
                band_stack.rio.to_raster(output_path, compress='LZW')
        except Exception as e:
            print(f"error, tile {idx} for year {year}: {e}")

    # Sentinel-2 data extraction
    bands = ['B02', 'B03', 'B04', 'B08']

    for idx, tile in tiles.iterrows():
        try:
            search = stac.search(
                collections=['sentinel-2-global-mosaics'],
                datetime=f"{year}-01-01T00:00:00Z/{year}-12-31T23:59:59Z",
                intersects=tile.geometry.__geo_interface__,
                limit=1000
            )

            items = search.item_collection()
            if len(items) == 0:
                print(f"no Sentinel-2 data found for tile {idx}, skip.")
                continue

            bounds_3059 = gpd.GeoSeries([tile.geometry], crs=4326).to_crs(3059).total_bounds.tolist()

            stack = stackstac.stack(
                items=items,
                assets=bands,
                resolution=10,
                bounds=bounds_3059,
                band_coords=True,
                snap_bounds=True,
                resampling=Resampling.bilinear,
                xy_coords='center',
                chunksize=1024,
                rescale=False,
                fill_value=0,
                dtype='int64',
                epsg=3059,
                gdal_env=gdal_env
            )

            stack['time'] = pd.to_datetime(stack['time'].values)

            for band in bands:
                output_path = os.path.join(outfolder_s2, f"cdse_s2_{band.lower()}_tile_{idx}.tif")
                if os.path.exists(output_path):
                    print(f"{idx} band {band} already exists, skip.")
                    continue
                band_stack = stack.sel(band=band)
                band_stack = band_stack.groupby('time.month').max(dim='time').transpose('month', 'y', 'x').compute()
                band_stack.rio.write_crs("EPSG:3059", inplace=True)
                band_stack.rio.to_raster(output_path, compress='LZW', dtype='int64')
        except Exception as e:
            print(f"error, tile {idx} for year {year}: {e}")


In [ ]:
# close the Dask client and cluster
client.close()
cluster.close()